# 5. Conclusion

## 5.1 Key results

The numbers below are read from the logs of the run when the book is built; chapters
[3](statistics.ipynb) and [4](performance.ipynb) give the details.

In [ ]:
import pandas as pd

import report_data as rd

rd.check()
stats, bench, papers = rd.stats(), rd.benchmark(), rd.papers()
status = stats["status"]
cov = stats["version_coverage"]
e0, e1 = stats["elements_before_dedup"], stats["elements_after_dedup"]
b0, b1 = stats["bib_entries_before_dedup"], stats["bib_entries_after_dedup"]
multi = papers[papers["versions_ok"] >= 2]
sessions = rd.sessions()
parsers = sessions["parser_version"].str.split("|").str[0]
first = sessions[parsers == parsers.iloc[0]]
if first["papers"].sum() >= stats["papers"]:   # one full pass with the first parser version
    wall, n_pass = first["wall_seconds"].sum(), int(first["papers"].sum())
else:
    wall, n_pass = bench["wall_seconds"], bench["papers"]
rd.md(f'''
- **Parser version `{stats["parser_version"]}`**: **{status.get("ok", 0):,} of {stats["versions_with_tex"]:,} versions
  with LaTeX source parsed ({stats["parse_success_rate_of_tex_versions_pct"]:.2f}%)**, {status.get("error", 0):,}
  errors; including empty-source versions (PDF-only, HTTP 404, unextractable),
  {stats["parse_success_rate_of_all_versions_pct"]:.2f}% of all {stats["versions"]:,} versions.
- **{cov["papers_full"]:,} of {stats["papers"]:,} papers fully covered** (every version parsed); mean coverage
  {cov["mean_pct"]:.2f}% of a paper's versions.
- **{e0:,} tree nodes became {e1:,} elements** ({rd.pct(e0 - e1, e0, 1)} fewer); for the
  {len(multi):,} papers with two or more parsed versions, {rd.pct(multi["nodes"].sum() - multi["elements"].sum(), multi["nodes"].sum(), 1)}
  of the nodes are repeated elements.
- **{b0:,} extracted BibTeX entries became {b1:,} rows** ({rd.pct(b0 - b1, b0, 1)} fewer).
- **{rd.hm(wall)} of wall time** for one full pass over the {n_pass:,} papers ({wall / n_pass:.3f} s per paper;
  {rd.hm(bench["wall_seconds"])} across all {bench["sessions"]} sessions including the targeted 13.8 and 13.9
  updates), **peak RAM {bench["peak_ram_mb"]:,.0f} MB**, output **{bench["output_bytes"] / rd.GB:.2f} GB**, on a
  CPU-only Colab runtime.
''')

## 5.2 What worked

- **A fixed interface between the four notebooks.** Each member wrote one step against an interface agreed at
  the start (chapter [1.3](overview.md)); notebook `13` loads the others with small adapters and keeps a
  fallback for any missing function, so the full pipeline ran end to end before every step was finished, and
  the parser version records which implementation produced each result.
- **Normalizing before matching.** Exact matching only works on text written one way. Expanding the macros,
  removing the layout commands, writing math one way and renaming the citation keys before the tree is built
  is what lets the unchanged parts of a paper match across versions (Figure 3.5).
- **Regression tests from real papers.** Every problem found in the corpus (a root in an included preamble,
  `\input` inside braces, an environment without its `\end`, files with carriage returns, nested macro
  definitions, REVTeX `\bibinfo` bibliography items) became a test before it was fixed, and every speed-up
  was checked to give byte-identical output on 900 papers.
- **Versioned carry-over updates.** Tagging every parsed paper with `parser_version` allowed targeted fixes
  (13.8 on 1,775 papers, 13.9 on 190 REVTeX papers) to re-parse only affected papers while verifying on
  random and manual-label samples that unaffected papers stayed byte-identical.

## 5.3 Difficulties

- **Papers that are not LaTeX documents.** Seven versions are Plain TeX (`\magnification`, `\bye`): they
  have `.tex` files but no `\documentclass`, so by the statement's definition they have no root.
- **Problems found only at full scale (13.8 and 13.9).** After the initial 13.7 full run, we fixed
  carriage-return line endings and nested macro removal in parser **13.8** (22 Nov, 1,775 flagged papers),
  and later fixed REVTeX `\bibitem` parsing (`\bibinfo{field}{...}`) in parser **13.9** (27 Nov, 190 papers
  with `\BibitemOpen`), eliminating 10,727 macro-corrupted titles/authors before locking the parser for
  Milestone 3 (chapter [2.5](running.md)).
- **References lost with the `.bbl` files.** Milestone 1 kept only `.tex` and `.bib` files, as its statement
  asked. Papers that cite a `.bib` they did not upload had their references only in the compiled `.bbl`, so
  those 2,231 versions (`missing_bbl` in Table 3.8) have no BibTeX entries.
- **Pathological sources.** Unclosed braces and math, recursive macros and bibliography files of tens of
  thousands of entries made early versions of the parser slow or stuck; each now has a linear fallback, and
  `15` bounds the time of every paper.

## 5.4 Limitations

- **A lexical parser, not TeX.** Macros with more than two parameters, an optional argument or `\def`
  parameters are not expanded, conditionals (`\if...\fi`) are not evaluated, and names defined by the document
  class stay as they are: the example of chapter [2.4](deduplication.ipynb) has a subsection titled
  `\ackname`.
- **Exact matching.** As the statement asks, two elements are merged only when their cleaned text is identical:
  a sentence in which the authors changed one word, or a heading whose capitalization changed, is a new
  element.
- **Sentence boundaries.** `split_sentences` also ends a sentence at `!` and `?`, and its list of
  abbreviations is finite; a period after an abbreviation it does not know ends the sentence.
- **References of `missing_bbl` and hand-written `none` versions.** The 2,231 `missing_bbl` versions could
  only be recovered by re-downloading their `.bbl` files; hand-written prose reference lists under
  `\section{References}` (part of the 496 `none` versions) are not scraped.